# 06 - Reusable Wrapper Layer, FastAPI, and Pydantic Validation

## Definition
Wrapper layer is software boundary that standardizes loading, validation, prediction, and metadata.

## Theory
Model artifact should never be called directly by external client. Wrapper controls input contract and error handling.

## Motivation
Without wrapper/API contract, model serving logic gets duplicated across services.


In [ ]:
from pathlib import Path
import os

CWD = Path.cwd()
ROOT = CWD if (CWD / "pyproject.toml").exists() else CWD.parent
os.chdir(ROOT)
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("outputs/benchmarks").mkdir(parents=True, exist_ok=True)


In [ ]:
from pathlib import Path
import numpy as np

from ml_package import ModelLoader, PredictionEngine
from ml_package.validation import IrisValidator

model_path = Path("models/iris_model.pkl")
engine = PredictionEngine(ModelLoader(model_path, verify_integrity=True).load())
validator = IrisValidator()


## Validation Layer Demonstration

### Valid request
Should pass with no errors.

### Invalid request
Should return detailed field-level feedback.


In [ ]:
valid_errors = validator.validate_request(5.1, 3.5, 1.4, 0.2)
invalid_errors = validator.validate_request(-1.0, 99.0, 0.0, -3.0)
print("Valid errors:", valid_errors)
print("Invalid errors:", invalid_errors)


In [ ]:
sample = np.array([[5.1, 3.5, 1.4, 0.2]])
pred = engine.predict(sample)
pred


## FastAPI Endpoint Walkthrough
- `GET /health`
- `GET /model-info`
- `POST /predict`
- `POST /predict-batch`
- `GET /metrics`
- `GET /metrics/prometheus`
- `POST /explain` (`mode=local|global`)


In [ ]:
from fastapi.testclient import TestClient
from api.main import app

client = TestClient(app)

health = client.get("/health")
print("Health:", health.status_code, health.json())

predict_payload = {
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2,
}
pred_resp = client.post("/predict", json=predict_payload)
print("Predict:", pred_resp.status_code, pred_resp.json())

metrics_resp = client.get("/metrics/prometheus")
print("Prometheus metrics snippet:\n", "\n".join(metrics_resp.text.splitlines()[:6]))
